In [ ]:
# ============================================================
# 🎮 VIDEO GAME DASHBOARD (PREMIUM DARK THEME)
# ============================================================

In [46]:
import pandas as pd
import numpy as np
import panel as pn
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio


In [47]:

# -------------------------
# PANEL EXTENSION + GLOBAL CSS
# -------------------------
pn.extension(
    'plotly',
    sizing_mode='stretch_width',
    raw_css=[
        """
        body {
            background: linear-gradient(135deg, #050f0d, #0f2e25);
        }

        .bk-root {
            background-color: transparent !important;
        }

        /* Cards */
        .card {
            background: rgba(20, 50, 40, 0.75);
            border-radius: 14px;
            padding: 15px;
            border: 1px solid rgba(0,255,150,0.2);
            box-shadow: 0 6px 25px rgba(0,255,150,0.08);
            backdrop-filter: blur(6px);
        }

        /* Sidebar */
        .bk-sidebar {
            background: #081c15 !important;
        }

        h1, h2, h3 {
            color: #00e676 !important;
        }
        """
    ]
)

In [49]:
# -------------------------
# PLOTLY THEME (NOT ONLY GREEN)
# -------------------------
pio.templates["dark_pro"] = dict(
    layout=dict(
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color="#d2f8e5"),

        # MULTI-COLOR (IMPORTANT)
        colorway=[
            "#00e676", "#00acc1", "#ffb300",
            "#ab47bc", "#ff7043", "#29b6f6"
        ]
    )
)

pio.templates.default = "dark_pro"


In [50]:
# -------------------------
# LOAD DATA
# -------------------------
df = pd.read_csv("cleaned_vgsales.csv")

In [52]:
# -------------------------
# FIX TYPES (important)
# -------------------------
df["Total Sales"] = pd.to_numeric(df["Total Sales"], errors="coerce")
df["Critic Score"] = pd.to_numeric(df["Critic Score"], errors="coerce")
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")


In [53]:
# -------------------------
# FILTERS
# -------------------------
console_filter = pn.widgets.Select(
    name="🎮 Console",
    options=["All"] + sorted(df["Console"].dropna().unique())
)

genre_filter = pn.widgets.MultiChoice(
    name="🎯 Genre",
    options=sorted(df["Genre"].dropna().unique())
)

year_filter = pn.widgets.IntRangeSlider(
    name="📅 Year",
    start=int(df["Year"].min()),
    end=int(df["Year"].max()),
    value=(int(df["Year"].min()), int(df["Year"].max()))
)

def filter_data(console, genres, year_range):
    dff = df.copy()

    if console != "All":
        dff = dff[dff["Console"] == console]

    if genres:
        dff = dff[dff["Genre"].isin(genres)]

    dff = dff[
        (dff["Year"] >= year_range[0]) &
        (dff["Year"] <= year_range[1])
    ]

    return dff

In [54]:
# ============================================================
# KPI CARDS
# ============================================================

def kpis(c, g, y):
    dff = filter_data(c, g, y)

    return pn.Row(

        pn.pane.Markdown(f"""
        <div class='card'>
        <h3>Total Games</h3>
        <h2>{len(dff)}</h2>
        </div>
        """),

        pn.pane.Markdown(f"""
        <div class='card'>
        <h3>Total Sales</h3>
        <h2>{round(dff["Total Sales"].sum(),2)}</h2>
        </div>
        """),

        pn.pane.Markdown(f"""
        <div class='card'>
        <h3>Avg Score</h3>
        <h2>{round(dff["Critic Score"].mean(),1)}</h2>
        </div>
        """),

        pn.pane.Markdown(f"""
        <div class='card'>
        <h3>Top Genre</h3>
        <h2>{dff["Genre"].mode()[0] if len(dff)>0 else "N/A"}</h2>
        </div>
        """),
    )

In [55]:
# ============================================================
# COMMON STYLE
# ============================================================

def style(fig):
    return fig.update_layout(
        margin=dict(l=10, r=10, t=40, b=10),
        plot_bgcolor='rgba(0,0,0,0)',
        paper_bgcolor='rgba(0,0,0,0)'
    )

In [56]:
# ============================================================
# CHARTS (1-5)
# ============================================================

def chart1(c,g,y):
    d = filter_data(c,g,y)
    return style(px.bar(
        d.sort_values("Total Sales", ascending=False).head(10),
        x="Total Sales", y="Title",
        orientation='h',
        title="Top 10 Games"
    ))

def chart2(c,g,y):
    d = filter_data(c,g,y)
    return style(px.pie(
        d, names="Genre", values="Total Sales",
        title="Genre Share"
    ))

def chart3(c,g,y):
    d = filter_data(c,g,y)
    return style(px.line(
        d.groupby("Year")["Total Sales"].sum().reset_index(),
        x="Year", y="Total Sales",
        title="Sales Trend"
    ))

def chart4(c,g,y):
    d = filter_data(c,g,y)
    r = d[["NA Sales","Japan Sales","PAL Sales","Other Sales"]].sum().reset_index()
    r.columns = ["Region","Sales"]
    return style(px.bar(r, x="Region", y="Sales", title="Regional Sales"))

def chart5(c,g,y):
    d = filter_data(c,g,y).dropna(subset=["Critic Score","Total Sales"])
    return style(px.scatter(
        d,
        x="Critic Score", y="Total Sales",
        size="Total Sales", color="Genre",
        title="Score vs Sales"
    ))


In [57]:
# ============================================================
# CHARTS (6-10)
# ============================================================

def chart6(c,g,y):
    d = filter_data(c,g,y)
    return style(px.bar(
        d.groupby("Console")["Total Sales"].sum().reset_index(),
        x="Console", y="Total Sales",
        title="Console Performance"
    ))

def chart7(c,g,y):
    d = filter_data(c,g,y)
    return style(px.box(
        d, x="Genre", y="Total Sales",
        title="Sales Distribution"
    ))

def chart8(c,g,y):
    d = filter_data(c,g,y)
    return style(px.histogram(
        d, x="Critic Score",
        title="Score Distribution"
    ))

def chart9(c,g,y):
    d = filter_data(c,g,y)
    return style(px.area(
        d.groupby("Year")["NA Sales"].sum().reset_index(),
        x="Year", y="NA Sales",
        title="NA Trend"
    ))

def chart10(c,g,y):
    d = filter_data(c,g,y)
    return style(px.bar(
        d.groupby("Publisher")["Total Sales"].sum()
        .sort_values(ascending=False).head(10).reset_index(),
        x="Publisher", y="Total Sales",
        title="Top Publishers"
    ))

In [61]:
# ============================================================
# DASHBOARD LAYOUT
# ============================================================

template = pn.template.FastListTemplate(
    title="🎮 Game Analytics Dashboard",
    header_background="#00c97f",

    sidebar=[
        pn.pane.Markdown("## 🎯 Filters"),
        console_filter,
        genre_filter,
        year_filter
    ],

    main=[

        pn.bind(kpis, console_filter, genre_filter, year_filter),

        pn.Row(
            pn.Column(pn.bind(chart1, console_filter, genre_filter, year_filter)),
            pn.Column(pn.bind(chart2, console_filter, genre_filter, year_filter))
        ),

        pn.Row(
            pn.Column(pn.bind(chart3, console_filter, genre_filter, year_filter)),
            pn.Column(pn.bind(chart4, console_filter, genre_filter, year_filter))
        ),

        pn.Row(
            pn.Column(pn.bind(chart5, console_filter, genre_filter, year_filter)),
            pn.Column(pn.bind(chart6, console_filter, genre_filter, year_filter))
        ),

        pn.Row(
            pn.Column(pn.bind(chart7, console_filter, genre_filter, year_filter)),
            pn.Column(pn.bind(chart8, console_filter, genre_filter, year_filter))
        ),

        pn.Row(
            pn.Column(pn.bind(chart9, console_filter, genre_filter, year_filter)),
            pn.Column(pn.bind(chart10, console_filter, genre_filter, year_filter))
        )
    ]
)
# template.servable()
# template

FastListTemplate
    [js_area] HTML(None, height=0, margin=0, sizing_mode='fixed', width=0)
    [actions] TemplateActions()
    [browser_info] BrowserInfo(dark_mode=True, device_pixel_ratio=1.25, language='en-US', timezone='Asia/Kolkata', timezone_offset=-330, webdriver=False, webgl=True)
    [busy_indicator] LoadingSpinner(height=20, width=20)
    [main-2424670251792] ParamFunction(function, _pane=Row, defer_load=False, sizing_mode='stretch_width')
    [main-2424744030768] Row(sizing_mode='stretch_width')
        [0] Column(sizing_mode='stretch_width')
            [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
        [1] Column(sizing_mode='stretch_width')
            [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
    [main-2424742797264] Row(sizing_mode='stretch_width')
        [0] Column(sizing_mode='stretch_width')
            [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
        [1] Column(sizing_mode='stretch_width')
            [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
    [main-2424742634320] Row(sizing_mode='stretch_width')
        [0] Column(sizing_mode='stretch_width')
            [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
        [1] Column(sizing_mode='stretch_width')
            [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
    [main-2424742614800] Row(sizing_mode='stretch_width')
        [0] Column(sizing_mode='stretch_width')
            [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
        [1] Column(sizing_mode='stretch_width')
            [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
    [main-2424742566320] Row(sizing_mode='stretch_width')
        [0] Column(sizing_mode='stretch_width')
            [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
        [1] Column(sizing_mode='stretch_width')
            [0] ParamFunction(function, _pane=Plotly, defer_load=False, sizing_mode='stretch_width')
    [nav-2424650168224] Markdown(str, sizing_mode='stretch_width')
    [nav-2424723994960] Select(name='🎮 Console', options=['All', '2600', ...], sizing_mode='stretch_width', value='All')
    [nav-2424723988576] MultiChoice(name='🎯 Genre', options=['Action', 'Action-Adventu...], sizing_mode='stretch_width')
    [nav-2424723988880] IntRangeSlider(end=2024, name='📅 Year', sizing_mode='stretch_width', start=1971, value=(1971, 2024), value_end=2024, value_start=1971)

In [60]:
pn.serve(template, show=True)

Launching server at http://localhost:60120


2026-04-23 18:10:56.203 200 GET / (127.0.0.1) 474.44ms
2026-04-23 18:10:56.269 200 GET /static/extensions/panel/bundled/plotlyplot/maplibre-gl@4.4.1/dist/maplibre-gl.css?v=1.8.10 (127.0.0.1) 1.85ms
2026-04-23 18:10:56.276 200 GET /static/extensions/panel/panel.min.js?v=b081f23dfb5efd1bbe84e7fda7427a25cfc354ce9fbf1fd8e5c35899acb2b3bb (127.0.0.1) 4.64ms
2026-04-23 18:10:56.279 200 GET /static/extensions/panel/bundled/reactiveesm/es-module-shims@%5E1.10.0/dist/es-module-shims.min.js (127.0.0.1) 6.70ms
2026-04-23 18:10:56.289 200 GET /static/js/bokeh-tables.min.js?v=62532f6ff926a41c47def08668ee2cec06b351a8e8a5ca01a21fdc78811c3fc5d6b048deaf949f6abc5c6963b7bbf56ea285d6ada0b7a61191449d0d115bf737 (127.0.0.1) 16.38ms
2026-04-23 18:10:56.293 200 GET /static/js/bokeh-widgets.min.js?v=aa4c33f38bd3425dafb970d1ce795ccf91e608a9a6d4db35f85f382954d7a91abf76031394f3af6ecb919a1b5dbbfefa2734c7a543132a8f0c0f24a2f9d3237b (127.0.0.1) 20.03ms
2026-04-23 18:10:56.301 200 GET /static/extensions/panel/bundled/@m

2026-04-23 18:11:07,282 ERROR: panel.reactive - Callback failed for object named '🎯 Genre' changing property {'value': ['Action-Adventure']} 
Traceback (most recent call last):
  File "C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\panel\reactive.py", line 479, in _process_events
    self.param.update(**self_params)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\param\parameterized.py", line 2788, in update
    restore = dict(self_._update(arg, **kwargs))
                   ~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\param\parameterized.py", line 2821, in _update
    self_._batch_call_watchers()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\param\parameterized.py", line 3038, in _batch_call_watchers
    self_._execute_watcher(watcher, events)
    ~~~~

2026-04-23 18:11:07.389 Exception in callback functools.partial(<bound method IOLoop._discard_future_result of <tornado.platform.asyncio.AsyncIOMainLoop object at 0x00000234CE094590>>, <Task finished name='Task-9564' coro=<ServerSession.with_document_locked() done, defined at C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\bokeh\server\session.py:77> exception=RuntimeError("Models must be owned by only a single document, ImportedStyleSheet(id='1384e763-cbb2-4782-9d58-c566881299b6', ...) is already in a doc")>)
Traceback (most recent call last):
  File "C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\tornado\ioloop.py", line 758, in _run_callback
    ret = callback()
  File "C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\tornado\ioloop.py", line 782, in _discard_future_result
    future.result()
    ~~~~~~~~~~~~~^^
  File "C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\bokeh\ser

2026-04-23 18:11:09,763 ERROR: panel.reactive - Callback failed for object named '🎯 Genre' changing property {'value': []} 
Traceback (most recent call last):
  File "C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\panel\reactive.py", line 479, in _process_events
    self.param.update(**self_params)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\param\parameterized.py", line 2788, in update
    restore = dict(self_._update(arg, **kwargs))
                   ~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\param\parameterized.py", line 2821, in _update
    self_._batch_call_watchers()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\param\parameterized.py", line 3038, in _batch_call_watchers
    self_._execute_watcher(watcher, events)
    ~~~~~~~~~~~~~~~~~~~~~~

2026-04-23 18:11:09.784 Exception in callback functools.partial(<bound method IOLoop._discard_future_result of <tornado.platform.asyncio.AsyncIOMainLoop object at 0x00000234CE094590>>, <Task finished name='Task-9850' coro=<ServerSession.with_document_locked() done, defined at C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\bokeh\server\session.py:77> exception=RuntimeError("Models must be owned by only a single document, ImportedStyleSheet(id='1384e763-cbb2-4782-9d58-c566881299b6', ...) is already in a doc")>)
Traceback (most recent call last):
  File "C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\tornado\ioloop.py", line 758, in _run_callback
    ret = callback()
  File "C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\tornado\ioloop.py", line 782, in _discard_future_result
    future.result()
    ~~~~~~~~~~~~~^^
  File "C:\Users\prakhar\AppData\Local\Programs\Python\Python314\Lib\site-packages\bokeh\ser